# CS 195: Natural Language Processing
## Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F6_2_Attention.ipynb)

## Reference

SLP: RNNs and LSTMs, Chapter 13 of Speech and Language Processing by Daniel Jurafsky & James H. Martin https://web.stanford.edu/~jurafsky/slp3/13.pdf


In [1]:
#import sys
#!{sys.executable} -m pip install torch datasets transformers

## Last Time: Encoder-Decoder Architecture

**Encoder RNN:** Takes input sequences, produces a context vector

**Context Vector:** Contains essence of the input sequence

**Decoder RNN:** Takes context vector as input, generates an output sequence

<div>
    <img src="images/encoder-decoder.png" width=700>
</div>


image source: SLP Fig. 13.16, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Encoder-Decoder with more detail

<div>
    <img src="images/encoder-decoder_detail.png" width=800>
</div>


image source: SLP Fig. 13.18, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Review Discussion

What kind of learning task do you use this for?

What goes into the $x$s? What goes into the $y$s?

## Started last time: A Synthetic Seq2Seq Task

Instead of jumping straight into a messy real-world task, let's use a small **synthetic** sequence-to-sequence problem.

**Task:** given a short sequence of tokens, predict the same sequence in reverse order.

Examples:
- input: `red dog runs`
- output: `runs dog red`

## Started last time: Build a Tiny Vocabulary and Synthetic Dataset

We will make short random sequences from a small vocabulary.

We'll also add special tokens:
- `<PAD>` for padding
- `<SOS>` for the start of decoder generation
- `<EOS>` for the end of the output sequence


In [2]:
import random

base_tokens = [
    'red', 'blue', 'green', 'yellow',
    'dog', 'cat', 'bird', 'fish',
    'runs', 'jumps', 'swims', 'sleeps',
    'canary', 'hamster', 'rabbit', 'turtle',
    'flies', 'hops', 'crawls', 'sings',
    'white', 'black', 'gray', 'brown',
    'car', 'bike', 'bus', 'train',
    'student', 'teacher', 'doctor', 'nurse',
    'eats', 'drinks', 'reads', 'writes'
]

special_tokens = ['<PAD>', '<SOS>', '<EOS>']
vocab = special_tokens + base_tokens

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

pad_id = word_to_id['<PAD>']
sos_id = word_to_id['<SOS>']
eos_id = word_to_id['<EOS>']
vocab_size = len(vocab)

print('vocab size:', vocab_size)
print(word_to_id)


def make_reverse_examples(num_examples=1000, min_len=3, max_len=5):
    inputs = []
    targets = []

    for _ in range(num_examples):
        seq_len = random.randint(min_len, max_len)
        tokens = random.sample(base_tokens, seq_len)
        reversed_tokens = list(reversed(tokens))
        inputs.append(tokens)
        targets.append(reversed_tokens)

    return inputs, targets


input_sequences, target_sequences = make_reverse_examples(min_len=3,max_len=5)
print(input_sequences[:3])
print(target_sequences[:3])


vocab size: 39
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, 'red': 3, 'blue': 4, 'green': 5, 'yellow': 6, 'dog': 7, 'cat': 8, 'bird': 9, 'fish': 10, 'runs': 11, 'jumps': 12, 'swims': 13, 'sleeps': 14, 'canary': 15, 'hamster': 16, 'rabbit': 17, 'turtle': 18, 'flies': 19, 'hops': 20, 'crawls': 21, 'sings': 22, 'white': 23, 'black': 24, 'gray': 25, 'brown': 26, 'car': 27, 'bike': 28, 'bus': 29, 'train': 30, 'student': 31, 'teacher': 32, 'doctor': 33, 'nurse': 34, 'eats': 35, 'drinks': 36, 'reads': 37, 'writes': 38}
[['hamster', 'dog', 'white', 'brown', 'car'], ['student', 'bus', 'sings', 'yellow'], ['train', 'hops', 'gray', 'nurse', 'flies']]
[['car', 'brown', 'white', 'dog', 'hamster'], ['yellow', 'sings', 'bus', 'student'], ['flies', 'nurse', 'gray', 'hops', 'train']]


## Started last time: Convert the Token Sequences to Training Tensors

For encoder-decoder training, we need three versions of the output sequence:
- the **encoder input**: the source sequence
- the **decoder input**: starts with `<SOS>` and then the target tokens
- the **decoder target**: the target tokens followed by `<EOS>`

This lets us use **teacher forcing** during training: at each decoder step, we give the decoder the correct previous token rather than its own prediction.


In [3]:
import torch

max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences) + 1  # room for <EOS>


def encode_input(seq):
    ids = [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_input_len - len(ids))
    return ids


def encode_decoder_input(seq):
    ids = [sos_id] + [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


def encode_decoder_target(seq):
    ids = [word_to_id[word] for word in seq] + [eos_id]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


encoder_inputs = torch.tensor([encode_input(seq) for seq in input_sequences], dtype=torch.long)
decoder_inputs = torch.tensor([encode_decoder_input(seq) for seq in target_sequences], dtype=torch.long)
decoder_targets = torch.tensor([encode_decoder_target(seq) for seq in target_sequences], dtype=torch.long)

print('encoder_inputs shape:', encoder_inputs.shape)
print('decoder_inputs shape:', decoder_inputs.shape)
print('decoder_targets shape:', decoder_targets.shape)
print('example encoder input:', encoder_inputs[0])
print('example decoder input:', decoder_inputs[0])
print('example decoder target:', decoder_targets[0])


encoder_inputs shape: torch.Size([1000, 5])
decoder_inputs shape: torch.Size([1000, 6])
decoder_targets shape: torch.Size([1000, 6])
example encoder input: tensor([16,  7, 23, 26, 27])
example decoder input: tensor([ 1, 27, 26, 23,  7, 16])
example decoder target: tensor([27, 26, 23,  7, 16,  2])


In [4]:
example_i = 0
print('source tokens:', input_sequences[example_i])
print('target tokens:', target_sequences[example_i])
print('decoder input tokens:', [id_to_word[i.item()] for i in decoder_inputs[example_i]])
print('decoder target tokens:', [id_to_word[i.item()] for i in decoder_targets[example_i]])


source tokens: ['hamster', 'dog', 'white', 'brown', 'car']
target tokens: ['car', 'brown', 'white', 'dog', 'hamster']
decoder input tokens: ['<SOS>', 'car', 'brown', 'white', 'dog', 'hamster']
decoder target tokens: ['car', 'brown', 'white', 'dog', 'hamster', '<EOS>']


## Started last time: Make Train/Test Splits and DataLoaders

We'll batch the encoder inputs, decoder inputs, and decoder targets together.


In [5]:
from torch.utils.data import TensorDataset, DataLoader

num_examples = len(encoder_inputs)
indices = list(range(num_examples))
random.shuffle(indices)

split = int(0.8 * num_examples)
train_idx = indices[:split]
test_idx = indices[split:]

train_dataset = TensorDataset(
    encoder_inputs[train_idx],
    decoder_inputs[train_idx],
    decoder_targets[train_idx],
)

# We're not actually going to validate with this dataset today
test_dataset = TensorDataset(
    encoder_inputs[test_idx],
    decoder_inputs[test_idx],
    decoder_targets[test_idx],
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

print('train examples:', len(train_dataset))
print('test examples:', len(test_dataset))


train examples: 800
test examples: 200


## Started last time: Encoder and Decoder in PyTorch

We'll test out a **GRU** instead of a plain **RNN**:
- it is still a recurrent model

The encoder reads the input sequence and returns a final hidden state.
The decoder starts from that hidden state and generates the output sequence one step at a time.


In [6]:
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        outputs, hidden = self.gru(emb)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, decoder_input_ids, hidden):
        emb = self.embedding(decoder_input_ids)
        outputs, hidden = self.gru(emb, hidden)
        logits = self.output(outputs)
        return logits, hidden


class Seq2SeqGRU(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim)

    def forward(self, encoder_input_ids, decoder_input_ids):
        encoder_outputs, hidden = self.encoder(encoder_input_ids)
        logits, hidden = self.decoder(decoder_input_ids, hidden)
        return logits


## New: Train with Teacher Forcing

During training, we already know the correct output sequence.
So instead of feeding the decoder its own previous prediction, we feed it the correct previous token from `decoder_inputs`.

This is called **teacher forcing**.
It usually makes training much easier.


In [7]:
import torch.optim as optim

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

model = Seq2SeqGRU(vocab_size).to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for enc_batch, dec_in_batch, dec_tgt_batch in train_loader:
        enc_batch = enc_batch.to(device)
        dec_in_batch = dec_in_batch.to(device)
        dec_tgt_batch = dec_tgt_batch.to(device) # [batch_size, target_len]

        logits = model(enc_batch, dec_in_batch)  # [batch_size, target_len, vocab_size]

        # logits are a 3D tensor now, so we need to reshape it for CrossEntropyLoss which wants a 2D tensor
        # reshape with -1 keeps the last dimension as vocab_size and flattens the rest
        logits_2D = logits.reshape(-1, vocab_size) # [batch_size * target_len, vocab_size]
        dec_tgt_batch_1D = dec_tgt_batch.reshape(-1)

        loss = loss_fn(logits_2D, dec_tgt_batch_1D )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        non_pad_tokens = (dec_tgt_batch != pad_id).sum().item() # count the number of non-pad tokens
        total_loss += loss.item() * non_pad_tokens
        total_tokens += non_pad_tokens

    avg_loss = total_loss / total_tokens
    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1}, avg token loss={avg_loss:.4f}')


Using device: mps
epoch 5, avg token loss=1.1324
epoch 10, avg token loss=0.2644
epoch 15, avg token loss=0.0972
epoch 20, avg token loss=0.0323
epoch 25, avg token loss=0.0143
epoch 30, avg token loss=0.0093


## Discussion Question

Think about the inputs to this model

`logits = model(enc_batch, dec_in_batch)`

Write down an example of what the *inputs* look like. Look at the data we prepared up above.
* What data is in `enc_batch`?
* What data is in `dec_in_batch`?

How is *teacher forcing* happening here?


## Greedy Inference

At inference time, we do **not** know the correct output sequence.
So we:
- encode the input once
- start the decoder with `<SOS>`
- predict one token
- feed that predicted token back into the decoder
- stop when we produce `<EOS>` or hit a maximum length


In [8]:
def decode_greedy(input_tokens, model, max_steps=max_target_len):
    # some layers behave differently during training/inference like Dropout layers
    # this is probably not doing anything in this example
    model.eval()

    encoder_ids = encode_input(input_tokens)
    encoder_tensor = torch.tensor([encoder_ids], dtype=torch.long).to(device)

    # .no_grad() tells PyTorch not to track gradients
    with torch.no_grad():

        # we only use the encoder part of the model!
        encoder_outputs, hidden = model.encoder(encoder_tensor)

        # Now to build the decoder input - we don't have a teacher forcing example to use
        # so let's start with the start-of-sequence token
        start_token_id = sos_id
        
        # Make a batch with one sequence containing just that one token
        # Shape: [batch_size=1, sequence_length=1]
        decoder_input = [[start_token_id]]
        
        decoder_input = torch.tensor(decoder_input, dtype=torch.long)
        
        decoder_input = decoder_input.to(device)
        
        generated_ids = []

        for _ in range(max_steps):
            logits, hidden = model.decoder(decoder_input, hidden)
            
            # Shape goes from [batch_size, sequence_length, vocab_size]
            # to [batch_size, vocab_size] and we only want the one from the last time step
            last_step_logits = logits[:, -1, :]
            
            # Pick the vocabulary item with the highest score for each example in the batch
            # Shape: [batch_size]
            predicted_token_ids = last_step_logits.argmax(dim=1)
            
            # Since we only have one example in the batch, pull out that one token id as a Python number
            next_id = predicted_token_ids.item()

            if next_id == eos_id:
                break

            # prepare the just-predicted token as the decoder input on the next step
            generated_ids.append(next_id)
            decoder_input = torch.tensor([[next_id]], dtype=torch.long).to(device)

    return [id_to_word[i] for i in generated_ids]


## Try the Model

The prediction should look like the source sequence in reverse order.


In [9]:
for i in range(5):
    source = input_sequences[test_idx[i]]
    target = target_sequences[test_idx[i]]
    prediction = decode_greedy(source, model)
    print('source:    ', source)
    print('target:    ', target)
    print('predicted: ', prediction)
    print()


source:     ['bike', 'canary', 'yellow', 'doctor', 'crawls']
target:     ['crawls', 'doctor', 'yellow', 'canary', 'bike']
predicted:  ['crawls', 'doctor', 'yellow', 'canary', 'runs']

source:     ['canary', 'reads', 'car']
target:     ['car', 'reads', 'canary']
predicted:  ['car', 'reads', 'canary']

source:     ['hamster', 'crawls', 'blue', 'nurse', 'student']
target:     ['student', 'nurse', 'blue', 'crawls', 'hamster']
predicted:  ['student', 'nurse', 'student', 'nurse', 'hamster', 'blue']

source:     ['runs', 'green', 'nurse', 'gray']
target:     ['gray', 'nurse', 'green', 'runs']
predicted:  ['gray', 'nurse', 'green', 'bike']

source:     ['blue', 'reads', 'brown', 'gray']
target:     ['gray', 'brown', 'reads', 'blue']
predicted:  ['gray', 'brown', 'reads', 'doctor']



## Discussion Question

For inference, we still predict one word at a time with the decoder

`logits, hidden = model.decoder(decoder_input, hidden)`

Where did `hidden` come from here? Is it the same value every time we predict a new word, or does it update over time?


## What This Helps Us See About Encoder-Decoder Models

This toy task is much simpler than translation or summarization, but it lets us see the core mechanics clearly:
- the **encoder** reads the input sequence into a hidden representation
- the **decoder** uses that representation to generate an output sequence
- during training we often use **teacher forcing**
- during inference we have to feed predictions back in one step at a time

This is the key bridge from language modeling to sequence-to-sequence learning.


## Applied Exploration

Start from this synthetic reverse-sequence task and try one or more of these extensions:

1. Increase the sequence length and see when the model starts to struggle.
2. Replace the `GRU` with an `LSTM` and compare the results.
3. Replace the reverse task with another synthetic task such as copying, sorting by a simple rule, or mapping digits to words.

Write up what you changed, why you changed it, and how the model's behavior changed.

## Extended Implementation Idea

There's a bit of work that needs to be done to train on a real dataset. Try a HuggingFace sequence-to-sequence dataset like
https://huggingface.co/datasets/abisee/cnn_dailymail



## Problem: Bottleneck

<div>
    <center>
    <img src="images/bottleneck.png" width=700>
    </center>
</div>

The final hidden state in the encoder has to contain everything meaningful about the input text
* may not represent things from earlier in the input sequence
* even if you use LSTM/GRU nodes

image source: SLP Fig. 13.20, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Attention

<div>
    <center>
    <img src="images/attention.png" width=800>
    </center>
</div>

**Idea:** instead of just passing the final hidden encoder state to the decoder, pass a combination of all encoder states
* weighted sum: makes sure that the context vector is a fixed size (can be some other more complicated function)
* computed again for each decoder state $i$
  - involves computing a score for the current decoder state with each encoder state
  - can be as simple as computing dot-product similarity
  - $\alpha_{i,j}$ is the softmax over all these scores
* takes into account decoder state $i-1$ and all encoder states
* you can learn *which input words are most important* when generating the next word
* even better at retaining long-term information

image source: SLP Fig. 13.22, https://web.stanford.edu/~jurafsky/slp3/13.pdf

## Explore changes needed to add an attention mechanism

**Step 1:** Use the same Encoder model, but we'll use a new Decoder

We use `torch.bmm` to do *batch matrix multiplication*

Each decoder output vector is being compared to each encoder output vector.

This is basically a bunch of dot products.

So:

decoder step i
encoder step j

get a score saying:

“how relevant is encoder position j when producing output position i?”
If the vectors point in similar directions, the dot product is larger.
If not, it is smaller.

In [10]:
class AttentionDecoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.gru = nn.GRU(embedding_dim, hidden_dim, batch_first=True)
        # self.output = nn.Linear(hidden_dim, vocab_size) # this was the old output layer without attention
        self.output = nn.Linear(hidden_dim * 2, vocab_size)

    # encoder_outputs is a new argument in this version
    # source_len is the input sequence length - like the length of the original input sequence in our dataset
    # target_len is the output sequence length - like the length of the reversed input sequence in our dataset
    def forward(self, decoder_input_ids, hidden, encoder_outputs):
        # same as before, but we also take in the encoder outputs so we can compute attention
        emb = self.embedding(decoder_input_ids)  # [batch, target_len, emb_dim]
        decoder_outputs, hidden = self.gru(emb, hidden)     # [batch, target_len, hidden_dim]

        # NEW CODE STARTS HERE
        # attention scores: compare each decoder output to each encoder output
        # encoder_outputs starts as [batch_size, source_len, hidden_dim]
        # encoder_outputs.transpose(1, 2) flips last two dimensions to [batch_size, hidden_dim, source_len]
        attention_scores = torch.bmm(decoder_outputs, encoder_outputs.transpose(1, 2))
        # [batch, target_len, source_len] - for each example, it's a table scoring how well each decoder output matches each encoder output

        # softmax to turn scores into probabilities along each row/target (over the source_len dimension)
        attention_weights = torch.softmax(attention_scores, dim=-1) 
        # [batch, target_len, source_len] - these are the attention weights that sum to 1 across the source_len dimension for each target token

        # weighted sum of encoder outputs
        context = torch.bmm(attention_weights, encoder_outputs)
        # [batch, target_len, hidden_dim]

        # both decoder_outputs and context have shape [batch, target_len, hidden_dim], so we can concatenate them along the last dimension
        # [batch, target_len, hidden_dim * 2]
        combined = torch.cat([decoder_outputs, context], dim=2)
        logits = self.output(combined) # [batch, target_len, vocab_size]

        #return logits, hidden, attention_weights
        return logits, hidden
    
class Seq2SeqAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim)
        self.decoder = AttentionDecoder(vocab_size, embedding_dim, hidden_dim)

    def forward(self, encoder_input_ids, decoder_input_ids):
        encoder_outputs, hidden = self.encoder(encoder_input_ids)
        logits, hidden = self.decoder(
            decoder_input_ids, hidden, encoder_outputs
        )
        #return logits, attention_weights
        return logits 
    
    # old version for comparison
    #def forward(self, encoder_input_ids, decoder_input_ids):
    #    encoder_outputs, hidden = self.encoder(encoder_input_ids)
    #    logits, hidden = self.decoder(decoder_input_ids, hidden)
    #    return logits


**Step 2:** change the training loop above to use this model



In [11]:
model = Seq2SeqAttention(vocab_size).to(device)


**Step 3:** 

Change your `decode_greedy` function so that you also pass the `encoder_outputs` to it

In [ ]:
# change this
logits, hidden = model.decoder(decoder_input, hidden)

# to this
logits, hidden = model.decoder(decoder_input, hidden, encoder_outputs)

## Creative Synthesis idea: Extended Implementation

There's a fair amount of work to be done to get this code working with a real learning task, though it can be done.

I suggest trying a summarization dataset like https://huggingface.co/datasets/abisee/cnn_dailymail or a text2emoji dataset like https://huggingface.co/datasets/KomeijiForce/Text2Emoji

Start with a small number of examples from the dataset and ramp up to whatever is computationally feasible.

Print a few examples and informally describe how well it performs. Or, perform a formal evaluation using ROUGE.